# CES Benchmark (Complex Exponential Smoothing) — smooth 1.1.1 (2026-10-01)

Rerun of the CES comparison on M1 + M3 + Tourism (5,315 series) with
**smooth 1.1.1**, against the previous smooth run of 2026-09-24 (smooth 1.0.9,
`2026-09-24-smooth-1-0-9-benchmark_auto_ces.ipynb`).

**Only the smooth rows were recomputed**: `AutoCES` with backcasting and optimal
initials (`CES Back`, `CES Opt`). There are no CSV files for CES in
`smooth-paper/Data`, so the `statsforecast` and `sktime` rows come from the saved
full-roster array `2026-08-22-benchmark-ces.npy`; it has no pinball. The previous
smooth rows come from `2026-09-24-benchmark-ces-smooth.npy` (no pinball either).

**No pinball for CES.** The Python `CES.predict()` accepts `interval=` but does not
compute prediction intervals yet (`lower`/`upper` come back as `None`), so only
RMSSE and SAME are reported.

**Timing caveat.** This run used 4 cores and Python 3.14, so times are not
comparable with the saved runs and are not discussed.

In [1]:
import os, glob, time, signal, datetime, subprocess, sys, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

HERE = os.getcwd()
DATA = os.path.join(HERE, "smooth-paper", "Data")
LEVELS = np.round(np.arange(0.01, 1.0, 0.01), 2)               # 99 quantile levels
TWO = [round(1 - 2 * t, 2) for t in LEVELS if t < 0.5]          # two-sided interval levels
VERSION_TAG = "smooth-1-1-1"
TASK_TIMEOUT = int(os.environ.get("BENCH_TASK_TIMEOUT", "120"))
DATE = "2026-10-01"
KIND = "ces"

## Versions

In [2]:
from importlib.metadata import version as pkg_version

print(f"python {sys.version.split()[0]}   cores {multiprocessing.cpu_count()}")
for _p in ("smooth", "greybox", "numpy", "pandas", "fcompdata"):
    print(f"{_p:12s} v{pkg_version(_p)}")
print("\ngit HEAD:", subprocess.run(["git", "log", "-1", "--format=%h %s"],
                                  capture_output=True, text=True).stdout.strip())

python 3.14.4   cores 4
smooth       v1.1.1
greybox      v1.0.7
numpy        v2.5.3
pandas       v3.0.6
fcompdata    v0.1.2

git HEAD: 92f3961e Fix sign-compare warnings in adamCore::reapply loops


## Datasets

In [3]:
from fcompdata import M1, M3, Tourism

datasets = ([M1[k] for k in M1.keys()]
            + [M3[k] for k in M3.keys()]
            + [Tourism[k] for k in Tourism.keys()])
print(f"M1: {len(M1)}   M3: {len(M3)}   Tourism: {len(Tourism)}   total: {len(datasets)}")

M1: 1001   M3: 3003   Tourism: 1311   total: 5315


## Methods and harness

In [4]:
METHODS = [
    ("CES Back", {"initial": "backcasting"}),
    ("CES Opt", {"initial": "optimal"}),
]
SMOOTH = [name for name, _ in METHODS]


def fit_predict(cfg, s):
    from smooth import AutoCES
    m = AutoCES(lags=lags_of(s), initial=cfg["initial"])
    m.fit(np.asarray(s.x, float))
    # CES.predict() returns no intervals yet, so CES is scored on the point forecast only.
    return np.asarray(m.predict(h=s.h).mean, float).ravel(), None

In [5]:
def lags_of(s):
    return [1, s.period] if s.period > 1 else [1]


def quantiles(fc):
    # The 99 quantiles from the two-sided intervals, with the mean at 0.5.
    mean = np.asarray(fc.mean, float).ravel()
    lower, upper = np.asarray(fc.lower, float), np.asarray(fc.upper, float)
    lo = {round(float(c), 2): i for i, c in enumerate(fc.lower.columns)}
    up = {round(float(c), 2): i for i, c in enumerate(fc.upper.columns)}
    Q = np.empty((mean.size, 99))
    for k, t in enumerate(LEVELS):
        if abs(t - 0.5) < 1e-9:
            Q[:, k] = mean
        elif t < 0.5:
            Q[:, k] = lower[:, lo[round(t, 2)]]
        else:
            Q[:, k] = upper[:, up[round(t, 2)]]
    return mean, Q


def score(s, mean, Q):
    # RMSSE, SAME and the scaled pinball averaged over the 99 levels.
    x, y = np.asarray(s.x, float), np.asarray(s.xx, float)
    scale_sq, scale_abs = np.mean(np.diff(x) ** 2), np.mean(np.abs(np.diff(x)))
    if not np.all(np.isfinite(mean)):
        return np.nan, np.nan, np.nan
    rmsse = np.sqrt(np.mean((y - mean) ** 2) / scale_sq) if scale_sq > 0 else np.nan
    same = np.abs(np.mean(y - mean)) / scale_abs if scale_abs > 0 else np.nan
    pinball = np.nan
    if Q is not None and np.all(np.isfinite(Q)):
        diff = y[:, None] - np.sort(Q, axis=1)
        loss = np.maximum(LEVELS * diff, (LEVELS - 1) * diff).mean(axis=0)
        pinball = np.mean(loss) / (scale_abs if scale_abs > 0 else 1.0)
    return rmsse, same, pinball


def _alarm(signum, frame):
    raise TimeoutError("task timeout")


def task(args):
    # One (method, series) fit: returns rmsse, same, time, pinball.
    warnings.filterwarnings("ignore")
    j, i, s, cfg = args
    out = np.full(4, np.nan)
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(TASK_TIMEOUT)
        start = time.time()
        mean, Q = fit_predict(cfg, s)
        out[2] = time.time() - start
        out[0], out[1], out[3] = score(s, mean[: s.h], None if Q is None else Q[: s.h])
    except Exception:
        pass
    finally:
        signal.alarm(0)
    return j, i, out


def run(methods, workers):
    result = np.full((len(methods), len(datasets), 4), np.nan)
    tasks = [(j, i, s, cfg) for j, (_, cfg) in enumerate(methods) for i, s in enumerate(datasets)]
    start = time.time()
    ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
        for done, f in enumerate(as_completed([ex.submit(task, t) for t in tasks]), 1):
            j, i, out = f.result()
            result[j, i] = out
            if done % 2000 == 0:
                print(f"  {done}/{len(tasks)}  {(time.time() - start) / 60:.1f} min", flush=True)
    print(f"done in {(time.time() - start) / 60:.1f} min")
    return result

## Run or load

In [6]:
# Fit the smooth rows on the current build, or load the saved 1.1.1 array
# (BENCH_RERUN=1 forces a refit). Columns: rmsse, same, time, pinball.
saved_file = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}.npy")
if os.environ.get("BENCH_RERUN", "0") == "1" or not os.path.exists(saved_file):
    workers = int(os.environ.get("BENCH_WORKERS", multiprocessing.cpu_count()))
    print(f"{len(METHODS)} methods x {len(datasets)} series, {workers} workers")
    current = run(METHODS, workers)
    np.save(saved_file, current)
else:
    current = np.load(saved_file)
print(os.path.basename(saved_file), current.shape)

2 methods x 5315 series, 4 workers


  2000/10630  1.2 min


  4000/10630  3.1 min


  6000/10630  7.1 min


  8000/10630  13.6 min


  10000/10630  28.6 min


done in 29.1 min
2026-10-01-benchmark-ces-smooth-1-1-1.npy (2, 5315, 4)


## Saved runs

In [7]:
# The current run, the previous smooth run and the competitors (saved files).
RESULT = {name: {"rmsse": current[j, :, 0], "same": current[j, :, 1],
                 "pinball": current[j, :, 3], "source": DATE}
          for j, name in enumerate(SMOOTH)}

prev = np.load("2026-09-24-benchmark-ces-smooth.npy")
prev_names = open("2026-09-24-benchmark-ces-smooth-methods.txt").read().split("\n")
PREVIOUS_LABEL = "2026-09-24 (smooth 1.0.9)"
PREVIOUS = {name: {"rmsse": prev[prev_names.index(name), :, 0],
                   "same": prev[prev_names.index(name), :, 1], "pinball": None}
            for name in SMOOTH}

saved_names = ["CES Back", "CES Opt", "CES Two", "CES none", "CES simple", "CES partial",
               "CES full", "statsforecast AutoCES", "sktime AutoCES"]
saved = np.load("2026-08-22-benchmark-ces.npy")
for name in ("statsforecast AutoCES", "sktime AutoCES"):
    RESULT[name] = {"rmsse": saved[saved_names.index(name), :, 0],
                    "same": saved[saved_names.index(name), :, 1],
                    "pinball": None, "source": "2026-08-22"}
for name in SMOOTH:
    RESULT[name + " (1.0.9)"] = dict(PREVIOUS[name], source="2026-09-24")

## Results summary

In [8]:
def metric_table(column, label):
    names = list(RESULT)
    values = {n: RESULT[n][column] for n in names}
    frame = pd.DataFrame({
        "Method": names,
        "Source": [RESULT[n]["source"] for n in names],
        "Min": [np.nanmin(values[n]) if values[n] is not None else np.nan for n in names],
        "Q1": [np.nanquantile(values[n], 0.25) if values[n] is not None else np.nan for n in names],
        "Med": [np.nanmedian(values[n]) if values[n] is not None else np.nan for n in names],
        "Q3": [np.nanquantile(values[n], 0.75) if values[n] is not None else np.nan for n in names],
        "Max": [np.nanmax(values[n]) if values[n] is not None else np.nan for n in names],
        "Mean": [np.nanmean(values[n]) if values[n] is not None else np.nan for n in names],
        "Failed": [int(np.sum(np.isnan(values[n]))) if values[n] is not None else np.nan
                   for n in names],
    })
    print("\n" + "=" * 110 + f"\n{label}, all {len(datasets)} series (lower = better)\n" + "=" * 110)
    print(frame.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return frame


summary_rmsse = metric_table("rmsse", "RMSSE")


RMSSE, all 5315 series (lower = better)
               Method     Source    Min     Q1    Med     Q3     Max   Mean  Failed
             CES Back 2026-10-01 0.0253 0.6711 1.1811 2.3163 51.1957 1.9607       0
              CES Opt 2026-10-01 0.0249 0.6795 1.2025 2.3135 49.1350 1.9889       0
statsforecast AutoCES 2026-08-22 0.0449 0.6781 1.1672 2.2861 50.6247 1.9573       0
       sktime AutoCES 2026-08-22 0.0449 0.6781 1.1672 2.2861 50.6247 1.9573       0
     CES Back (1.0.9) 2026-09-24 0.0248 0.8268 1.3655 2.3946 51.2724 2.0784       0
      CES Opt (1.0.9) 2026-09-24 0.0249 0.8552 1.3922 2.4082 49.1350 2.1179       0


In [9]:
summary_same = metric_table("same", "SAME")


SAME, all 5315 series (lower = better)
               Method     Source    Min     Q1    Med     Q3     Max   Mean  Failed
             CES Back 2026-10-01 0.0002 0.3777 0.9973 2.3908 54.9099 2.0008       0
              CES Opt 2026-10-01 0.0001 0.4011 1.0202 2.4093 54.8930 2.0395       0
statsforecast AutoCES 2026-08-22 0.0004 0.3706 1.0009 2.3995 54.8929 1.9978       0
       sktime AutoCES 2026-08-22 0.0004 0.3706 1.0009 2.3995 54.8929 1.9978       0
     CES Back (1.0.9) 2026-09-24 0.0001 0.3860 1.0146 2.4122 54.9071 1.9998       0
      CES Opt (1.0.9) 2026-09-24 0.0003 0.4048 1.0304 2.4458 54.8930 2.0480       0


In [10]:
summary_pinball = metric_table("pinball", "Scaled pinball (mean over 99 levels)")


Scaled pinball (mean over 99 levels), all 5315 series (lower = better)
               Method     Source  Min  Q1  Med  Q3  Max  Mean    Failed
             CES Back 2026-10-01  NaN NaN  NaN NaN  NaN   NaN 5315.0000
              CES Opt 2026-10-01  NaN NaN  NaN NaN  NaN   NaN 5315.0000
statsforecast AutoCES 2026-08-22  NaN NaN  NaN NaN  NaN   NaN       NaN
       sktime AutoCES 2026-08-22  NaN NaN  NaN NaN  NaN   NaN       NaN
     CES Back (1.0.9) 2026-09-24  NaN NaN  NaN NaN  NaN   NaN       NaN
      CES Opt (1.0.9) 2026-09-24  NaN NaN  NaN NaN  NaN   NaN       NaN


## Change against the previous smooth run

The same series, before (smooth 1.0.9) and after (smooth 1.1.1). `Better`/`Worse` count the series where the metric went down/up.

In [11]:
def compare(column):
    rows = []
    for name in SMOOTH:
        before, after = PREVIOUS[name][column], RESULT[name][column]
        row = {"Method": name, "Metric": column.upper()}
        for tag, v in (("before", before), ("after", after)):
            row[f"Mean {tag}"] = np.nanmean(v) if v is not None else np.nan
            row[f"Med {tag}"] = np.nanmedian(v) if v is not None else np.nan
        if before is not None:
            d = after - before
            row["Better"], row["Worse"] = int(np.sum(d < -1e-8)), int(np.sum(d > 1e-8))
        rows.append(row)
    return rows


rows = [r for column in ("rmsse", "same", "pinball") for r in compare(column)]
print(f"before = {PREVIOUS_LABEL}   after = {DATE} (smooth {pkg_version('smooth')})")
change = pd.DataFrame(rows)
print(change.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

before = 2026-09-24 (smooth 1.0.9)   after = 2026-10-01 (smooth 1.1.1)
  Method  Metric  Mean before  Med before  Mean after  Med after    Better     Worse
CES Back   RMSSE       2.0784      1.3655      1.9607     1.1811 3195.0000 2120.0000
 CES Opt   RMSSE       2.1179      1.3922      1.9889     1.2025 1470.0000  345.0000
CES Back    SAME       1.9998      1.0146      2.0008     0.9973 2615.0000 2700.0000
 CES Opt    SAME       2.0480      1.0304      2.0395     1.0202  914.0000  807.0000
CES Back PINBALL          NaN         NaN         NaN        NaN       NaN       NaN
 CES Opt PINBALL          NaN         NaN         NaN        NaN       NaN       NaN
